# Agentic RAG — let the model pick the retrieval strategy

Naive RAG always runs a search. Self-RAG / CRAG add quality gates. **Agentic RAG** goes one step further: the model picks the *retrieval strategy* itself via tool calls.

Three tools mirror three real branches you'd implement in production:

1. `search_papers(query, k)` — semantic search.
2. `lookup_doc(doc_id)` — direct fetch when the user names a document.
3. `answer_directly(answer)` — no retrieval needed.

The notebook walks one example per route.

In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
import json
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, ToolSpec, complete
from shared.loaders import load_corpus, load_golden_qa

MODEL = 'openai/gpt-4o-mini'
NS = '01-rag/08-agentic-rag'
DOCS = load_corpus()
QA = {q.id: q for q in load_golden_qa()}
DOC_BY_ID = {d.arxiv_id: d for d in DOCS}
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
doc_vecs = hash_embed(doc_texts, dims=256, seed=0)

## Tool specs

In [3]:
TOOLS = [
    ToolSpec(name='search_papers', description='Semantic search over the arxiv-cs.CL synthetic corpus. Returns top doc ids.',
             parameters={'type':'object','properties':{'query':{'type':'string'},'k':{'type':'integer','default':3}},'required':['query']}),
    ToolSpec(name='lookup_doc', description='Return the full abstract for a single doc id (e.g. \'synth-001\').',
             parameters={'type':'object','properties':{'doc_id':{'type':'string'}},'required':['doc_id']}),
    ToolSpec(name='answer_directly', description='Use when the question needs no retrieval (greeting, definition of a generic term).',
             parameters={'type':'object','properties':{'answer':{'type':'string'}},'required':['answer']}),
]
AGENT_SYS = (
    'You are a research assistant with three tools: search_papers, lookup_doc, answer_directly. '
    'First pick the SINGLE best tool for the user\'s question, then after seeing its result give '
    'a concise final answer grounded in the tool output. Never invent citations.'
)

## Tool implementations

In [4]:
def t_search_papers(query: str, k: int = 3):
    qv = hash_embed([query], dims=256, seed=0)[0]
    idx, _ = cosine_topk(qv, doc_vecs, k=k)
    return [{'doc_id': doc_ids[i], 'snippet': doc_texts[i][:120]} for i in idx]

def t_lookup_doc(doc_id: str):
    d = DOC_BY_ID[doc_id]
    return {'doc_id': doc_id, 'title': d.title, 'abstract': d.abstract}

def t_answer_directly(answer: str):
    return answer

DISPATCH = {'search_papers': t_search_papers, 'lookup_doc': t_lookup_doc, 'answer_directly': t_answer_directly}

## One-step agent loop

Model picks tool → we execute → feed result back → model answers.

In [5]:
def run_agent(user_q):
    msgs = [Message(role='system', content=AGENT_SYS),
            Message(role='user', content=user_q)]
    first = complete(model=MODEL, namespace=NS, messages=msgs, tools=TOOLS)
    if not first.tool_calls:
        return {'route': 'no-tool', 'answer': first.content}
    call = first.tool_calls[0]
    args = json.loads(call.arguments)
    result = DISPATCH[call.name](**args)
    tool_payload = result if isinstance(result, str) else json.dumps(result)
    msgs_after = msgs + [
        Message(role='assistant', content=''),
        Message(role='tool', name=call.name, content=tool_payload),
    ]
    final = complete(model=MODEL, namespace=NS, messages=msgs_after)
    return {'route': call.name, 'tool_args': args, 'tool_result': result, 'answer': final.content}

## Three examples — one per route

In [6]:
examples = [
    QA['q01'].question,
    'Give me the abstract of synth-005.',
    'What does the abbreviation RAG stand for in NLP?',
]
for q in examples:
    out = run_agent(q)
    print('Q:', q)
    print('  route :', out['route'])
    print('  answer:', out['answer'])
    print()

Q: By how much does RA-MoE reduce p99 decode latency compared to standard learned routing?
  route : search_papers
  answer: RA-MoE has 47B total parameters with two experts active per token, giving ~13B active params (synth-001).

Q: Give me the abstract of synth-005.
  route : lookup_doc
  answer: synth-005 is 'Tokenizer Equity: Measuring and Mitigating Length Disparities Across 104 Languages'. It introduces the Tokenizer-Equity Index (TEI) for cross-lingual fairness.

Q: What does the abbreviation RAG stand for in NLP?
  route : answer_directly
  answer: Retrieval-Augmented Generation.



## What you can extend

* Add a `cite(doc_id, span)` tool and require the final answer to include at least one.
* Multi-step: allow the agent to make 2-3 sequential tool calls (search → lookup → answer).
* Add a budget cap on tool calls per query to prevent runaway loops.
* Combine with CRAG: after a `search_papers` call, run the grader and possibly re-route.